# Prepare the closed-set benchmark

Build observation-aware gallery and query manifests for four external datasets and the five paper seeds. The canonical image catalogues also feed the separate AnimalCLEF workflow.

## Inputs

Supply the foreground-processed images and their manifests. Use the configuration cell to point to the mounted data. The private jaguar overlap audit requires the original ApexID corpus metadata. See [the evaluation guide](../docs/evaluation.md) for artifact layouts.

## Repository and environment

In [ ]:
import json
import os
import sys
from pathlib import Path

REPO_ROOT = Path(os.environ.get("APEXID_REPO_ROOT", Path.cwd())).resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "apexid").is_dir():
    raise FileNotFoundError("Start the notebook in the repository or set APEXID_REPO_ROOT.")
sys.path.insert(0, str(REPO_ROOT))

import shutil
import subprocess
from IPython.display import display

WORKSPACE_ROOT = Path(os.environ.get("APEXID_OUTPUT_ROOT", REPO_ROOT / "outputs")).resolve()

## 2. Minimal preparation runtime

This cell is derived from the current CVPR benchmark runtime but contains only the dependency closure needed for dataset preparation, hashing/grouping, Split V2, and split validation.

In [ ]:
from __future__ import annotations

import base64

import contextlib

import dataclasses

import hashlib

import importlib.util

import io

import json

import os

import platform

import random

import subprocess

import sys

import tempfile

import time

import zlib

from collections import Counter, defaultdict

from dataclasses import dataclass, field

from pathlib import Path

from typing import Any, Callable, Mapping, Sequence

import numpy as np

import pandas as pd

from PIL import Image

from scipy.fft import dctn

BENCHMARK_VERSION = "apexid-cvpr-benchmark-1.0.1"

DEFAULT_SPLIT_SEEDS = (42, 123, 3407, 2026, 1337)

def _json_default(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if dataclasses.is_dataclass(value):
        return dataclasses.asdict(value)
    if isinstance(value, set):
        return sorted(value)
    raise TypeError(f"Cannot serialize {type(value).__name__}")

def stable_json(value: Any) -> str:
    return json.dumps(
        value,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
        default=_json_default,
    )

def sha256_bytes(value: bytes) -> str:
    return hashlib.sha256(value).hexdigest()

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            block = handle.read(chunk_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()

def sha256_frame(frame: pd.DataFrame, columns: Sequence[str] | None = None) -> str:
    selected = frame if columns is None else frame.loc[:, list(columns)]
    payload = selected.to_csv(index=False, lineterminator="\n").encode("utf-8")
    return sha256_bytes(payload)

def atomic_write_text(path: Path, text: str) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temporary_name = tempfile.mkstemp(prefix=f".{path.name}.", dir=path.parent)
    os.close(fd)
    temporary = Path(temporary_name)
    try:
        temporary.write_text(text, encoding="utf-8")
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)

def write_json(path: Path, value: Any) -> None:
    atomic_write_text(Path(path), json.dumps(value, indent=2, sort_keys=True, default=_json_default) + "\n")

def atomic_write_csv(path: Path, frame: pd.DataFrame) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temporary_name = tempfile.mkstemp(prefix=f".{path.name}.", dir=path.parent)
    os.close(fd)
    temporary = Path(temporary_name)
    try:
        frame.to_csv(temporary, index=False, lineterminator="\n")
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)

@dataclass
class RuntimeRecorder:
    rows: list[dict[str, Any]] = field(default_factory=list)

    @contextlib.contextmanager
    def measure(self, stage: str, **metadata: Any):
        start = time.perf_counter()
        try:
            yield
        finally:
            row = {
                "stage": stage,
                "seconds": time.perf_counter() - start,
                **metadata,
            }
            self.rows.append(row)

    def record(self, stage: str, seconds: float, **metadata: Any) -> None:
        self.rows.append({
            "stage": stage,
            "seconds": float(seconds),
            **metadata,
        })

    def frame(self) -> pd.DataFrame:
        base = [
            "stage",
            "model",
            "dataset",
            "split_seed",
            "seconds",
            "cache_hit",
            "number_images",
            "number_queries",
            "number_gallery_identities",
        ]

        frame = pd.DataFrame(self.rows)

        for column in base:
            if column not in frame:
                frame[column] = np.nan

        return frame[
            base + [c for c in frame.columns if c not in base]
        ]

    def save(self, path: Path) -> None:
        atomic_write_csv(path, self.frame())

def stable_seed(*parts: Any) -> int:
    digest = hashlib.sha256("::".join(map(str, parts)).encode("utf-8")).digest()
    return int.from_bytes(digest[:8], byteorder="big", signed=False) % (2**32 - 1)

def seed_everything(seed: int) -> None:
    random.seed(int(seed))
    np.random.seed(int(seed))
    try:
        import torch

        torch.manual_seed(int(seed))
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(int(seed))
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
    except ImportError:
        pass

def _package_version(name: str) -> str | None:
    try:
        from importlib.metadata import version

        return version(name)
    except Exception:
        return None

def environment_provenance() -> dict[str, Any]:
    result: dict[str, Any] = {
        "python": sys.version,
        "platform": platform.platform(),
        "os": os.name,
        "packages": {
            name: _package_version(name)
            for name in (
                "numpy",
                "pandas",
                "scipy",
                "scikit-learn",
                "torch",
                "torchvision",
                "timm",
                "transformers",
                "wildlife-datasets",
                "albumentations",
                "opencv-python-headless",
                "Pillow",
                "huggingface-hub",
                "safetensors",
            )
        },
    }
    try:
        import torch

        result["torch"] = {
            "version": torch.__version__,
            "cuda_version": torch.version.cuda,
            "cudnn_version": torch.backends.cudnn.version(),
            "cuda_available": torch.cuda.is_available(),
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        }
    except ImportError:
        result["torch"] = {"available": False}
    return result

INVALID_IDENTITIES = {"", "unknown", "none", "nan", "null", "new_whale"}

EXPLICIT_GROUP_COLUMNS = (
    "encounter_id",
    "observation_id",
    "camera_event_id",
    "event_id",
    "sequence_id",
    "video_id",
    "burst_id",
    "track_id",
    "sighting_id",
)

TEMPORAL_COLUMNS = ("timestamp", "datetime", "captured_at", "capture_time", "date")

CAMERA_COLUMNS = ("camera_id", "camera", "station_id", "location_id")

def normalize_external_frame(
    raw: pd.DataFrame,
    *,
    dataset: str,
    data_root: Path,
    path_column: str = "path",
    identity_column: str = "identity",
    species: str | None = None,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    if path_column not in raw or identity_column not in raw:
        raise ValueError(
            f"{dataset}: manifest must contain {identity_column!r} and {path_column!r}; "
            f"found {list(raw.columns)}"
        )
    frame = raw.copy()
    raw_count = len(frame)
    frame["identity"] = frame[identity_column].astype(str).str.strip()
    invalid = frame["identity"].str.lower().isin(INVALID_IDENTITIES)
    removed_invalid = int(invalid.sum())
    frame = frame.loc[~invalid].copy()
    root = Path(data_root)
    resolved: list[str] = []
    missing: list[str] = []
    for value in frame[path_column].astype(str):
        candidate = Path(value)
        if not candidate.is_absolute():
            candidate = root / candidate
        candidate = candidate.resolve()
        if not candidate.is_file():
            missing.append(str(candidate))
        resolved.append(str(candidate))
    if missing:
        raise FileNotFoundError(
            f"{dataset}: {len(missing)} manifest images are missing. Examples: {missing[:5]}"
        )
    frame["path"] = resolved
    frame["dataset"] = str(dataset)
    if "species" not in frame:
        frame["species"] = species or "unknown"
    else:
        frame["species"] = frame["species"].fillna(species or "unknown").astype(str)
    frame = frame.sort_values(["identity", "path"], kind="mergesort").reset_index(drop=True)
    image_ids = [
        f"{dataset}::{hashlib.sha1(path.encode('utf-8')).hexdigest()[:20]}"
        for path in frame["path"].astype(str)
    ]
    if len(image_ids) != len(set(image_ids)):
        duplicates = frame.loc[pd.Series(image_ids).duplicated(keep=False), "path"].tolist()
        raise ValueError(f"{dataset}: repeated image paths in cleaned manifest: {duplicates[:5]}")
    frame.insert(0, "image_id", image_ids)
    audit = {
        "dataset": dataset,
        "raw_records": int(raw_count),
        "records_removed": int(removed_invalid),
        "removal_reasons": {"invalid_or_unknown_identity": removed_invalid},
        "final_records_before_group_eligibility": int(len(frame)),
        "final_identities_before_group_eligibility": int(frame["identity"].nunique()),
        "dataset_source": str(data_root),
        "preprocessing": "manifest-defined source image",
        "segmentation_background_removal": "not declared",
        "overlap_removal_count": 0,
    }
    return frame, audit

def load_merged_dataset(
    merged_csv: Path,
    dataset: str,
    *,
    data_root: Path | None = None,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    merged_csv = Path(merged_csv)
    if not merged_csv.is_file():
        raise FileNotFoundError(f"Merged external manifest missing: {merged_csv}")
    raw = pd.read_csv(merged_csv)
    if "dataset" not in raw:
        raise ValueError(f"Merged manifest lacks dataset column: {merged_csv}")
    selected = raw.loc[raw["dataset"].astype(str) == dataset].copy()
    if selected.empty:
        raise ValueError(f"Merged manifest contains no rows for dataset={dataset!r}")
    return normalize_external_frame(
        selected,
        dataset=dataset,
        data_root=data_root or merged_csv.parent,
    )

def load_bristol_sam3(manifest_path: Path, dataset_root: Path) -> tuple[pd.DataFrame, dict[str, Any]]:
    manifest_path = Path(manifest_path)
    if not manifest_path.is_file():
        raise FileNotFoundError(f"Bristol SAM3 manifest missing: {manifest_path}")
    raw = pd.read_csv(manifest_path)
    frame, audit = normalize_external_frame(
        raw,
        dataset="bristol_gorillas",
        data_root=dataset_root,
        path_column="segmented_image",
        identity_column="identity",
        species="western_lowland_gorilla",
    )
    audit["preprocessing"] = "SAM3 foreground with black background"
    audit["segmentation_background_removal"] = "required: SAM3 black background"
    return frame, audit

def load_jaguar_antonio(
    manifest_path: Path,
    *,
    dataset_root: Path,
    apexid_metadata_csv: Path,
    require_black_background: bool = True,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    manifest_path = Path(manifest_path)
    if not manifest_path.is_file():
        raise FileNotFoundError(f"Jaguar Antonio prepared manifest missing: {manifest_path}")
    raw = pd.read_csv(manifest_path, dtype={"identity": str, "jaguar_id": str})
    identity_column = "identity" if "identity" in raw else "jaguar_id"
    path_column = next((name for name in ("path", "segmented_image", "filepath") if name in raw), None)
    if path_column is None:
        raise ValueError("Jaguar Antonio manifest has no path/segmented_image/filepath column")
    if require_black_background:
        if "segmented_black_bg" not in raw:
            raise ValueError(
                "Jaguar Antonio manifest must explicitly declare segmented_black_bg for every row"
            )
        declared = raw["segmented_black_bg"].astype(str).str.lower().isin({"true", "1", "yes"})
        if not bool(declared.all()):
            raise ValueError("Jaguar Antonio contains rows not declared as black-background composites")
    before_overlap = len(raw)
    train_ids: set[str] = set()
    apexid_metadata_csv = Path(apexid_metadata_csv)
    if not apexid_metadata_csv.is_file():
        raise FileNotFoundError(
            "ApexID metadata is required before Jaguar overlap filtering: "
            f"{apexid_metadata_csv}"
        )
    training = pd.read_csv(apexid_metadata_csv)
    if "identity" not in training:
        raise ValueError("ApexID metadata lacks identity column")
    mask = pd.Series(False, index=training.index)
    if "dataset" in training:
        mask |= training["dataset"].astype(str).str.lower().isin({"jaguar_data", "jaguar"})
    if "species" in training:
        mask |= training["species"].astype(str).str.lower().str.contains("jaguar", regex=False)
    train_ids = set(training.loc[mask, "identity"].astype(str).str.strip().str.lower())
    raw_identity = raw[identity_column].astype(str).str.strip().str.lower()
    overlap_mask = raw_identity.isin(train_ids)
    raw = raw.loc[~overlap_mask].copy()
    frame, audit = normalize_external_frame(
        raw,
        dataset="jaguar_antonio",
        data_root=dataset_root,
        path_column=path_column,
        identity_column=identity_column,
        species="jaguar",
    )
    overlap_count = int(before_overlap - len(raw))
    audit["raw_records"] = int(before_overlap)
    audit["records_removed"] += overlap_count
    audit["removal_reasons"]["apexid_training_identity_overlap"] = overlap_count
    audit["overlap_removal_count"] = overlap_count
    audit["preprocessing"] = "FiftyOne body-mask foreground on black background"
    audit["segmentation_background_removal"] = "required and validated"
    return frame, audit

def _dhash(image: Image.Image) -> int:
    pixels = np.asarray(image.convert("L").resize((9, 8), Image.Resampling.BILINEAR), dtype=np.uint8)
    bits = pixels[:, 1:] > pixels[:, :-1]
    return int.from_bytes(np.packbits(bits).tobytes(), "big")

def _phash(image: Image.Image) -> int:
    pixels = np.asarray(image.convert("L").resize((32, 32), Image.Resampling.BILINEAR), dtype=np.float32)
    low = dctn(pixels, type=2, norm="ortho")[:8, :8]
    median = float(np.median(low.reshape(-1)[1:]))
    return int.from_bytes(np.packbits(low > median).tobytes(), "big")

def fingerprint_images(frame: pd.DataFrame, *, workers: int = 0) -> pd.DataFrame:
    """Compute exact and perceptual hashes once, in deterministic manifest order."""
    required = {"image_id", "path", "identity", "dataset"}
    if not required <= set(frame.columns):
        raise ValueError(f"Image frame missing columns: {sorted(required - set(frame.columns))}")

    def calculate(record: tuple[int, str]) -> tuple[int, str, str, str, int, int]:
        index, path_text = record
        path = Path(path_text)
        exact = sha256_file(path)
        with Image.open(path) as image:
            image.load()
            width, height = image.size
            return index, exact, f"{_dhash(image):016x}", f"{_phash(image):016x}", width, height

    records = list(enumerate(frame["path"].astype(str).tolist()))
    if workers and workers > 1:
        from concurrent.futures import ThreadPoolExecutor

        with ThreadPoolExecutor(max_workers=int(workers)) as executor:
            values = list(executor.map(calculate, records))
    else:
        values = [calculate(record) for record in records]
    values.sort(key=lambda item: item[0])
    result = frame.copy().reset_index(drop=True)
    result["image_sha256"] = [item[1] for item in values]
    result["dhash"] = [item[2] for item in values]
    result["phash"] = [item[3] for item in values]
    result["image_width"] = [item[4] for item in values]
    result["image_height"] = [item[5] for item in values]
    conflicting = (
        result.groupby("image_sha256", sort=False)[["dataset", "identity"]]
        .agg(lambda values_: tuple(sorted(set(map(str, values_)))))
        .reset_index()
    )
    conflicts = conflicting.loc[
        conflicting["identity"].map(len).gt(1) | conflicting["dataset"].map(len).gt(1)
    ]
    if not conflicts.empty:
        raise ValueError(
            "Exact image bytes occur under conflicting dataset/identity labels: "
            f"{conflicts.head(5).to_dict(orient='records')}"
        )
    return result

class UnionFind:
    def __init__(self, size: int):
        self.parent = list(range(size))
        self.rank = [0] * size

    def find(self, value: int) -> int:
        parent = self.parent[value]
        if parent != value:
            self.parent[value] = self.find(parent)
        return self.parent[value]

    def union(self, first: int, second: int) -> None:
        left, right = self.find(first), self.find(second)
        if left == right:
            return
        if self.rank[left] < self.rank[right]:
            left, right = right, left
        self.parent[right] = left
        if self.rank[left] == self.rank[right]:
            self.rank[left] += 1

def _hamming(first: str, second: str) -> int:
    return (int(first, 16) ^ int(second, 16)).bit_count()

def perceptual_candidate_pairs(
    frame: pd.DataFrame,
    *,
    dhash_threshold: int = 4,
    phash_threshold: int = 6,
) -> list[tuple[int, int, int, int]]:
    """Find all conservative near-duplicate pairs without quadratic full scans.

    Eight 8-bit pHash bands guarantee that any pair with total pHash Hamming
    distance <= 6 shares at least one band.  Candidate pairs are then checked
    against both pHash and dHash thresholds.
    """
    pairs: list[tuple[int, int, int, int]] = []
    buckets: dict[tuple[int, int], list[int]] = defaultdict(list)
    candidates: set[tuple[int, int]] = set()
    for index, phash_text in zip(frame.index, frame["phash"].astype(str)):
        value = int(phash_text, 16)
        for band in range(8):
            key = (band, (value >> (band * 8)) & 0xFF)
            for previous in buckets[key]:
                candidates.add((min(int(previous), int(index)), max(int(previous), int(index))))
            buckets[key].append(int(index))
    for first, second in sorted(candidates):
        ph = _hamming(str(frame.at[first, "phash"]), str(frame.at[second, "phash"]))
        if ph > int(phash_threshold):
            continue
        dh = _hamming(str(frame.at[first, "dhash"]), str(frame.at[second, "dhash"]))
        if dh <= int(dhash_threshold):
            pairs.append((first, second, dh, ph))
    return pairs

def _usable_column(frame: pd.DataFrame, candidates: Sequence[str], minimum_coverage: float) -> str | None:
    for column in candidates:
        if column not in frame:
            continue
        values = frame[column].astype(str).str.strip()
        valid = ~values.str.lower().isin(INVALID_IDENTITIES)
        if float(valid.mean()) >= float(minimum_coverage) and values[valid].nunique() > 1:
            return column
    return None

def build_observation_groups(
    frame: pd.DataFrame,
    *,
    minimum_metadata_coverage: float = 0.95,
    encounter_gap_minutes: float = 30.0,
    dhash_threshold: int = 4,
    phash_threshold: int = 6,
) -> tuple[pd.DataFrame, dict[str, Any], pd.DataFrame]:
    required = {"image_id", "identity", "dataset", "image_sha256", "dhash", "phash"}
    if not required <= set(frame):
        raise ValueError(f"Observation grouping missing columns: {sorted(required - set(frame))}")
    result = frame.reset_index(drop=True).copy()
    union = UnionFind(len(result))
    # ------------------------------------------------------------
    # Exact byte-identical images may never have conflicting labels
    # ------------------------------------------------------------
    
    exact_identity_counts = (
        result
        .groupby("image_sha256")["identity"]
        .nunique()
    )
    
    conflicting_exact_hashes = exact_identity_counts[
        exact_identity_counts > 1
    ].index.tolist()
    
    if conflicting_exact_hashes:
    
        examples = []
    
        for digest in conflicting_exact_hashes[:5]:
    
            rows = result[
                result["image_sha256"] == digest
            ]
    
            examples.append({
                "sha256": digest,
                "identities": sorted(
                    rows["identity"].astype(str).unique().tolist()
                ),
                "image_ids": rows[
                    "image_id"
                ].astype(str).tolist()[:10],
            })
    
        raise ValueError(
            "Exact byte-identical images have conflicting identity "
            f"labels. Manual dataset correction required: {examples}"
        )
    for (_identity, _digest), indices in result.groupby(["identity", "image_sha256"], sort=True).groups.items():
        indices = sorted(map(int, indices))
        for index in indices[1:]:
            union.union(indices[0], index)

    explicit_column = _usable_column(result, EXPLICIT_GROUP_COLUMNS, minimum_metadata_coverage)
    temporal_column = None if explicit_column else _usable_column(result, TEMPORAL_COLUMNS, minimum_metadata_coverage)
    camera_column = _usable_column(result, CAMERA_COLUMNS, 0.5)
    strategy_parts: list[str] = []

    if explicit_column:
        strategy_parts.append(f"explicit encounter: {explicit_column}")
        valid = ~result[explicit_column].astype(str).str.strip().str.lower().isin(INVALID_IDENTITIES)
        for (_identity, _value), indices in result.loc[valid].groupby(["identity", explicit_column], sort=True).groups.items():
            indices = sorted(map(int, indices))
            for index in indices[1:]:
                union.union(indices[0], index)
    elif temporal_column:
        parsed = pd.to_datetime(result[temporal_column], errors="coerce", utc=True)
        result["observation_timestamp"] = parsed
        if parsed.isna().any():
            raise ValueError(
                f"{result['dataset'].iloc[0]}: temporal grouping selected {temporal_column!r}, "
                f"but {int(parsed.isna().sum())} rows are unparseable"
            )
        strategy_parts.append(f"date/time grouping: {temporal_column}")
        grouping_columns = ["identity"] + ([camera_column] if camera_column else [])
        for _key, indices in result.groupby(grouping_columns, sort=True).groups.items():
            ordered = sorted(map(int, indices), key=lambda idx: (parsed.iloc[idx], result.at[idx, "image_id"]))
            for previous, current in zip(ordered, ordered[1:]):
                delta = (parsed.iloc[current] - parsed.iloc[previous]).total_seconds() / 60.0
                same_date = parsed.iloc[current].date() == parsed.iloc[previous].date()
                is_date_only = all(
                    len(str(result.at[idx, temporal_column]).strip()) <= 10 for idx in (previous, current)
                )
                if (is_date_only and same_date) or (not is_date_only and delta <= encounter_gap_minutes):
                    union.union(previous, current)

    near_pairs = perceptual_candidate_pairs(
        result,
        dhash_threshold=dhash_threshold,
        phash_threshold=phash_threshold,
    )
    # ------------------------------------------------------------
    # Separate same-identity near duplicates from cross-identity
    # perceptual-similarity candidates.
    #
    # Same identity:
    #   safe to use as a must-link observation constraint.
    #
    # Different identity:
    #   DO NOT merge automatically. Perceptual hashes measure
    #   image similarity, not animal identity. Preserve these as
    #   hard negatives and record them for auditing.
    # ------------------------------------------------------------
    
    same_identity_near_pairs = []
    cross_identity_near_pairs = []
    
    for first, second, dh, ph in near_pairs:
    
        first_identity = str(result.at[first, "identity"])
        second_identity = str(result.at[second, "identity"])
    
        if first_identity == second_identity:
            same_identity_near_pairs.append(
                (first, second, dh, ph)
            )
        else:
            cross_identity_near_pairs.append(
                (first, second, dh, ph)
            )
    
    
    # Only same-identity perceptual matches become must-links.
    for first, second, _dh, _ph in same_identity_near_pairs:
        union.union(first, second)
    
    
    if cross_identity_near_pairs:
    
        examples = [
            {
                "first_image_id": result.at[first, "image_id"],
                "first_identity": result.at[first, "identity"],
                "second_image_id": result.at[second, "image_id"],
                "second_identity": result.at[second, "identity"],
                "dhash_distance": int(dh),
                "phash_distance": int(ph),
            }
            for first, second, dh, ph
            in cross_identity_near_pairs[:10]
        ]
    
        print(
            f"[AUDIT] {result['dataset'].iloc[0]}: "
            f"{len(cross_identity_near_pairs)} cross-identity "
            "perceptual-similarity candidates detected. "
            "They are NOT merged."
        )
    
        print("Examples:")
        for example in examples:
            print(example)
    if same_identity_near_pairs or (not explicit_column and not temporal_column):
        strategy_parts.append(
            "visual similarity fallback: "
            "same-identity exact+dHash+pHash must-link"
        )

    members: dict[int, list[int]] = defaultdict(list)
    for index in range(len(result)):
        members[union.find(index)].append(index)
    group_id_by_index: dict[int, str] = {}
    for indices in members.values():
        dataset = str(result.at[indices[0], "dataset"])
        identity = str(result.at[indices[0], "identity"])
        if any(str(result.at[idx, "identity"]) != identity for idx in indices):
            raise ValueError("Observation must-link component crosses identities")
        digest = hashlib.sha1(
            "\n".join(sorted(result.loc[indices, "image_id"].astype(str))).encode("utf-8")
        ).hexdigest()[:16]
        group_id = f"{dataset}::{identity}::obs::{digest}"
        for index in indices:
            group_id_by_index[index] = group_id
    result["observation_group_id"] = [group_id_by_index[index] for index in range(len(result))]
    result["observation_group_strategy"] = " + ".join(strategy_parts)

    pair_rows = [
        {
            "first_image_id": result.at[first, "image_id"],
            "second_image_id": result.at[second, "image_id"],
            "identity": result.at[first, "identity"],
            "dhash_distance": dh,
            "phash_distance": ph,
            "observation_group_id": result.at[first, "observation_group_id"],
        }
        for first, second, dh, ph in same_identity_near_pairs
    ]
    pair_frame = pd.DataFrame(
        pair_rows,
        columns=[
            "first_image_id",
            "second_image_id",
            "identity",
            "dhash_distance",
            "phash_distance",
            "observation_group_id",
        ],
    )
    diagnostics = {
        "strategy": result["observation_group_strategy"].iloc[0] if len(result) else "none",
        "explicit_column": explicit_column,
        "temporal_column": temporal_column,
        "camera_column": camera_column,
        "images": int(len(result)),
        "identities": int(result["identity"].nunique()),
        "observation_groups": int(result["observation_group_id"].nunique()),
        "exact_duplicate_groups": int((result.groupby("image_sha256").size() > 1).sum()),
        "near_duplicate_pairs": int(len(pair_frame)),
        "dhash_threshold": int(dhash_threshold),
        "phash_threshold": int(phash_threshold),
        "cross_identity_perceptual_candidates":int(len(cross_identity_near_pairs)),
    }
    return result, diagnostics, pair_frame

def dataset_manifest_hash(frame: pd.DataFrame) -> str:
    columns = [
        "image_id",
        "path",
        "identity",
        "dataset",
        "species",
        "observation_group_id",
        "image_sha256",
        "dhash",
        "phash",
    ]
    return sha256_frame(frame.sort_values("image_id", kind="mergesort"), columns)

def _stable_shuffle(values: Sequence[str], *seed_parts: Any) -> list[str]:
    ordered = sorted(map(str, values))
    generator = np.random.default_rng(stable_seed(*seed_parts))
    if ordered:
        ordered = [ordered[index] for index in generator.permutation(len(ordered))]
    return ordered

def build_external_split_manifest(
    frame: pd.DataFrame,
    *,
    split_seed: int,
    gallery_group_ratio: float = 0.80,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    if not 0.0 < gallery_group_ratio < 1.0:
        raise ValueError("gallery_group_ratio must be strictly between 0 and 1")
    required = {"image_id", "identity", "observation_group_id", "image_sha256", "dataset"}
    if not required <= set(frame):
        raise ValueError(f"Split input missing columns: {sorted(required - set(frame))}")
    manifest = frame.copy()
    group_counts = manifest.groupby("identity")["observation_group_id"].nunique()
    eligible = group_counts[group_counts >= 2].index.astype(str)
    if len(eligible) < 2:
        raise ValueError("Closed-set retrieval requires at least two eligible identities")
    dataset = str(manifest["dataset"].iloc[0])
    roles = {}
    for identity, rows in manifest.groupby("identity"):
        if str(identity) not in eligible:
            continue
        groups = _stable_shuffle(
            rows["observation_group_id"].astype(str).unique(),
            "gallery-groups", dataset, split_seed, str(identity),
        )
        n_gallery = min(max(int(round(len(groups) * gallery_group_ratio)), 1), len(groups) - 1)
        gallery_groups = set(groups[:n_gallery])
        for image_id, group in zip(rows["image_id"].astype(str), rows["observation_group_id"].astype(str)):
            roles[image_id] = "gallery" if group in gallery_groups else "query_known"
    manifest["eligible"] = manifest["identity"].astype(str).isin(eligible)
    manifest["eligibility_reason"] = np.where(
        manifest["eligible"], "eligible", "fewer_than_two_observation_groups"
    )
    manifest["closed_role"] = manifest["image_id"].astype(str).map(roles).fillna("excluded")
    manifest["split_seed"] = int(split_seed)
    manifest = manifest.sort_values("image_id", kind="mergesort").reset_index(drop=True)
    return manifest, split_diagnostics(manifest)


def split_diagnostics(manifest: pd.DataFrame) -> dict[str, Any]:
    active = manifest.loc[manifest["closed_role"].ne("excluded")]
    gallery = active.loc[active["closed_role"].eq("gallery")]
    query = active.loc[active["closed_role"].eq("query_known")]
    return {
        "dataset": str(manifest["dataset"].iloc[0]),
        "split_seed": int(manifest["split_seed"].iloc[0]),
        "number_images": int(len(active)),
        "number_identities": int(active["identity"].nunique()),
        "number_observation_groups": int(active["observation_group_id"].nunique()),
        "number_gallery_identities": int(gallery["identity"].nunique()),
        "number_known_query_identities": int(query["identity"].nunique()),
        "number_gallery_samples": int(len(gallery)),
        "number_known_query_samples": int(len(query)),
    }


def validate_external_split(
    manifest: pd.DataFrame,
    near_duplicate_pairs: pd.DataFrame | None = None,
) -> dict[str, Any]:
    active = manifest.loc[manifest["closed_role"].ne("excluded")]
    gallery = active.loc[active["closed_role"].eq("gallery")]
    query = active.loc[active["closed_role"].eq("query_known")]
    if manifest["image_id"].duplicated().any() or gallery.empty or query.empty:
        raise ValueError("Closed-set split has duplicate records or an empty gallery/query role")
    if set(query["identity"]) - set(gallery["identity"]):
        raise ValueError("Closed-set query identities must occur in the gallery")
    if set(gallery["observation_group_id"]) & set(query["observation_group_id"]):
        raise ValueError("Observation groups cross closed-set roles")
    if (set(gallery["image_sha256"]) & set(query["image_sha256"])) - {""}:
        raise ValueError("Exact image duplicates cross closed-set roles")
    if near_duplicate_pairs is not None:
        role_by_image = dict(zip(manifest["image_id"], manifest["closed_role"]))
        for row in near_duplicate_pairs.itertuples(index=False):
            if {role_by_image[row.first_image_id], role_by_image[row.second_image_id]} == {"gallery", "query_known"}:
                raise ValueError("Detected within-identity near-duplicates cross closed-set roles")
    return {"passed": True, **split_diagnostics(manifest)}



def enforce_train_val_must_link_closure(
    rows: Sequence[dict[str, Any]],
    splits: Mapping[str, Sequence[dict[str, Any]]],
    *,
    component_splits: Sequence[str] = (
        "train",
        "val_known",
        "val_unknown",
        "val_unknown_extra",
        "unlabeled_ood",
    ),
) -> tuple[dict[str, list[dict[str, Any]]], dict[str, Any]]:
    """Conservatively close train/val_known must-link crossings.

    Split V2's explicit-source assignment is identity-local, while explicit group identifiers can
    be shared by multiple identities. When that occurs, the repository validator correctly rejects
    the crossing. Moving affected validation rows to train is the only one-way repair that cannot
    leak a training observation into validation or turn a known identity into an unknown identity.
    The closure repeats because moving one row can expose a crossing through another must-link key.
    """
    del splits  # Row annotations are authoritative; rebuilding avoids stale membership lists.
    guard_fields = (
        "_image_sha256",
        "_duplicate_group_id",
        "_explicit_group_id",
        "_similarity_cluster_id",
        "_observation_group_id",
    )
    moved_indices: set[int] = set()
    reason_counts: Counter = Counter()
    rounds = 0

    def rebuild() -> dict[str, list[dict[str, Any]]]:
        result = {
            name: sorted(
                [row for row in rows if str(row.get("_split", "")) == name],
                key=lambda row: int(row["_idx"]),
            )
            for name in component_splits
        }
        result["val"] = result["val_known"] + result["val_unknown"]
        return result

    for _iteration in range(len(rows) + 1):
        to_move: dict[int, set[str]] = defaultdict(set)
        for group_field in guard_fields:
            groups: dict[str, list[dict[str, Any]]] = defaultdict(list)
            for row in rows:
                value = str(row.get(group_field, ""))
                if value:
                    groups[value].append(row)
            for members in groups.values():
                sides = {str(row.get("_split", "")) for row in members}
                if {"train", "val_known"} <= sides:
                    for row in members:
                        if str(row.get("_split", "")) == "val_known":
                            to_move[int(row["_idx"])].add(group_field)

        date_groups: dict[tuple[Any, str], list[dict[str, Any]]] = defaultdict(list)
        for row in rows:
            date = str(row.get("_parsed_date", ""))
            if date:
                date_groups[(row.get("_key"), date)].append(row)
        for members in date_groups.values():
            sides = {str(row.get("_split", "")) for row in members}
            if {"train", "val_known"} <= sides:
                for row in members:
                    if str(row.get("_split", "")) == "val_known":
                        to_move[int(row["_idx"])].add("identity_date")

        if not to_move:
            repaired = rebuild()
            if not repaired["val_known"]:
                raise RuntimeError(
                    "Must-link closure removed every val_known row; independent calibration is impossible"
                )
            return repaired, {
                "applied": bool(moved_indices),
                "rounds": rounds,
                "moved_val_known_rows_to_train": len(moved_indices),
                "reason_counts": dict(sorted(reason_counts.items())),
                "policy": "conservative one-way val_known-to-train must-link closure",
            }

        rounds += 1
        by_index = {int(row["_idx"]): row for row in rows}
        for index, reasons in sorted(to_move.items()):
            row = by_index[index]
            if str(row.get("_split")) != "val_known":
                continue
            row["_split"] = "train"
            row["_split_method"] = str(row.get("_split_method", "")) + "+must_link_closure_train"
            moved_indices.add(index)
            reason_counts.update(reasons)
    raise RuntimeError("Must-link closure did not converge")

def build_split_v2_with_must_link_repair(
    *,
    repo_root: Path,
    metadata_csv: Path,
    data_root: Path,
    output_dir: Path,
    feature_cache: Path,
    verify_images: bool = False,
    near_duplicate_audit: bool = True,
) -> None:
    """Run the official CLI with a local assignment hook before its integrity check.

    All paths are absolute: the notebook's working directory need not be the repository.
    The CLI is loaded in its own namespace; neither repository files nor notebook globals
    are overwritten. Its feature cache, validators, manifest and atomic writer are reused.
    """
    repo_root = Path(repo_root).resolve()
    script = repo_root / "data" / "make_open_set_splits.py"
    spec = importlib.util.spec_from_file_location("_apexid_cvpr_split_builder", script)
    if spec is None or spec.loader is None:
        raise ImportError(f"Cannot load the official Split V2 builder: {script}")
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))
    cli = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(cli)
    original_builder = cli.make_observation_aware_splits
    original_summary = cli.build_summary
    repair: dict[str, Any] = {}

    def repaired_builder(rows, *args, **kwargs):
        tentative = original_builder(rows, *args, **kwargs)
        original_counts = {key: len(value) for key, value in tentative.items()}
        repaired, diagnostics = enforce_train_val_must_link_closure(
            rows, tentative, component_splits=cli.COMPONENT_SPLITS
        )
        repair.update(diagnostics)
        repair["original_rows"] = original_counts
        repair["repaired_rows"] = {key: len(value) for key, value in repaired.items()}
        if diagnostics["applied"]:
            print(
                "Split V2 source-group repair: moved "
                f"{diagnostics['moved_val_known_rows_to_train']} conflicting validation rows "
                "to train; validating the resulting split.",
                flush=True,
            )
        return repaired

    def summary_with_repair(*args, **kwargs):
        summary = original_summary(*args, **kwargs)
        summary["benchmark_split_repair"] = {
            **repair,
            "benchmark_version": BENCHMARK_VERSION,
            "official_script_sha256": sha256_file(script),
            "official_split_core_sha256": sha256_file(repo_root / "apexid/data/split_v2.py"),
        }
        return summary

    cli.make_observation_aware_splits = repaired_builder
    cli.build_summary = summary_with_repair
    arguments = [
        "--config", str(repo_root / "data" / "split_v2.yaml"),
        "--metadata", str(Path(metadata_csv).resolve()),
        "--data-root", str(Path(data_root).resolve()),
        "--output-dir", str(Path(output_dir).resolve()),
        "--feature-cache", str(Path(feature_cache).resolve()),
    ]
    if verify_images:
        arguments.append("--verify-images")
    if near_duplicate_audit:
        arguments.append("--near-duplicate-audit")
    try:
        cli.main(arguments)
    finally:
        cli.make_observation_aware_splits = original_builder
        cli.build_summary = original_summary

def ensure_and_validate_split_v2(
    *,
    repo_root: Path,
    metadata_csv: Path,
    data_root: Path,
    output_dir: Path,
    feature_cache: Path,
    auto_build: bool = True,
    verify_images: bool = False,
    near_duplicate_audit: bool = True,
) -> dict[str, Any]:
    repo_root = Path(repo_root).resolve()
    metadata_csv = Path(metadata_csv).resolve()
    data_root = Path(data_root).resolve()
    output_dir = Path(output_dir).resolve()
    feature_cache = Path(feature_cache).resolve()
    required = (
        "train.csv",
        "val_known.csv",
        "val_unknown.csv",
        "val_unknown_extra.csv",
        "unlabeled_ood.csv",
        "val.csv",
        "split_manifest.csv",
        "split_summary.json",
    )
    if not metadata_csv.is_file():
        raise FileNotFoundError(f"ApexID authoritative metadata missing: {metadata_csv}")
    missing = [name for name in required if not (output_dir / name).is_file()]
    invalid_reason = f"missing artifacts: {missing}" if missing else None
    if not missing:
        try:
            result = _validate_split_v2_artifacts(
                repo_root, metadata_csv, data_root, output_dir,
                verify_images=verify_images, near_duplicate_audit=near_duplicate_audit,
            )
        except (ValueError, RuntimeError, KeyError) as exc:
            if not auto_build:
                raise
            invalid_reason = str(exc)
        else:
            print(f"CACHE HIT — verified ApexID Split V2: {output_dir}", flush=True)
            return result
    if not auto_build:
        raise FileNotFoundError(f"Split V2 is unavailable ({invalid_reason}): {output_dir}")

    # Build/validate in a staging directory. Only a verified result can replace the target.
    # Preserve existing artifacts in a sibling backup so a rerun after failure loses no work.
    protected = (repo_root, data_root, metadata_csv.parent, feature_cache, Path.cwd().resolve())
    if output_dir == Path(output_dir.anchor) or any(
        output_dir == path or output_dir in path.parents for path in protected
    ):
        raise ValueError(f"Split V2 output must be a dedicated artifact directory: {output_dir}")
    if output_dir.exists() and not output_dir.is_dir():
        raise ValueError(f"Split V2 output is not a directory: {output_dir}")
    print(f"Building ApexID Split V2 ({invalid_reason}); reusing {feature_cache}", flush=True)
    output_dir.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix=f".{output_dir.name}.build-", dir=output_dir.parent) as stage:
        staged_output = Path(stage) / "split"
        try:
            build_split_v2_with_must_link_repair(
                repo_root=repo_root, metadata_csv=metadata_csv, data_root=data_root,
                output_dir=staged_output, feature_cache=feature_cache,
                verify_images=verify_images, near_duplicate_audit=near_duplicate_audit,
            )
            result = _validate_split_v2_artifacts(
                repo_root, metadata_csv, data_root, staged_output,
                verify_images=verify_images, near_duplicate_audit=near_duplicate_audit,
            )
        except Exception:
            if staged_output.is_dir():
                diagnostic_dir = Path(tempfile.mkdtemp(
                    prefix=f".{output_dir.name}.failed-", dir=output_dir.parent
                )) / "split"
                os.replace(staged_output, diagnostic_dir)
                print(f"Failed split and leakage diagnostics preserved at: {diagnostic_dir}", flush=True)
            raise
        backup = None
        if output_dir.exists():
            backup = Path(tempfile.mkdtemp(prefix=f".{output_dir.name}.backup-", dir=output_dir.parent))
            backup = backup / output_dir.name
            os.replace(output_dir, backup)
        try:
            os.replace(staged_output, output_dir)
        except Exception:
            if backup is not None:
                os.replace(backup, output_dir)
            raise
        if backup is not None:
            print(f"Previous Split V2 artifacts preserved at: {backup}", flush=True)
            result["previous_artifacts_backup"] = str(backup)
    return result

def _validate_split_v2_artifacts(
    repo_root: Path, metadata_csv: Path, data_root: Path, output_dir: Path,
    *, verify_images: bool, near_duplicate_audit: bool,
) -> dict[str, Any]:

    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))
    from apexid.data.split_v2 import validate_written_v2

    official = validate_written_v2(
        output_dir,
        metadata_csv,
        data_root=data_root,
        verify_images=verify_images,
    )
    manifest = pd.read_csv(output_dir / "split_manifest.csv", dtype=str).fillna("")
    expected_columns = {
        "canonical_identity",
        "image_sha256",
        "split",
        "split_method",
        "observation_group_id",
        "split_seed",
        "feature_model",
        "feature_revision",
    }
    if not expected_columns <= set(manifest):
        raise RuntimeError(
            "Split manifest is not Split V2; missing columns "
            f"{sorted(expected_columns - set(manifest))}"
        )
    methods = sorted(set(manifest["split_method"].astype(str)) - {""})
    if not methods:
        raise RuntimeError("Split V2 manifest contains no split_method provenance")
    train = pd.read_csv(output_dir / "train.csv", dtype=str).fillna("")
    known = pd.read_csv(output_dir / "val_known.csv", dtype=str).fillna("")
    unknown = pd.read_csv(output_dir / "val_unknown.csv", dtype=str).fillna("")
    if train.empty or known.empty or unknown.empty:
        raise RuntimeError("Split V2 calibration requires nonempty train, val_known and val_unknown")
    # Repository identities are dataset-local, even when their printed labels coincide.
    train_ids = set(zip(train["dataset"], train["identity"]))
    known_ids = set(zip(known["dataset"], known["identity"]))
    unknown_ids = set(zip(unknown["dataset"], unknown["identity"]))
    if train_ids & unknown_ids:
        raise RuntimeError("Split V2 train and val_unknown identities overlap")
    if known_ids - train_ids:
        raise RuntimeError("Split V2 val_known identities are absent from train")
    group_crossings = []
    for field_name in ("image_sha256", "observation_group_id", "duplicate_group_id", "explicit_group_id", "similarity_cluster_id"):
        if field_name not in manifest:
            continue
        for value, rows in manifest.loc[manifest[field_name] != ""].groupby(field_name):
            splits = set(rows["split"].astype(str))
            if "train" in splits and "val_known" in splits:
                group_crossings.append((field_name, value))
                break
    if group_crossings:
        raise RuntimeError(f"Split V2 observation/similarity groups cross train/val_known: {group_crossings[:5]}")
    summary = json.loads((output_dir / "split_summary.json").read_text(encoding="utf-8"))
    if summary.get("schema_version") != 2 or summary.get("metadata", {}).get("sha256") != sha256_file(metadata_csv):
        raise RuntimeError("Split V2 summary does not match the authoritative metadata and schema")
    for filename, expected_hash in summary.get("output_hashes", {}).items():
        if Path(filename).name != filename or not (output_dir / filename).is_file():
            raise RuntimeError(f"Invalid or missing Split V2 artifact: {filename}")
        if sha256_file(output_dir / filename) != expected_hash:
            raise RuntimeError(f"Split V2 artifact hash mismatch: {filename}")
    if not bool(summary.get("integrity", {}).get("passed", False)):
        raise RuntimeError("Split V2 split_summary.json does not record integrity.passed=true")
    near_info = summary.get("near_duplicate_audit", {})
    if near_duplicate_audit and not near_info.get("enabled", False):
        raise RuntimeError(
            "Split V2 exists but its repository near-duplicate audit was not enabled. "
            "Rebuild Split V2 with --near-duplicate-audit or explicitly disable this sealed check."
        )
    return {
        "passed": True,
        "official_validation": official,
        "split_manifest_sha256": sha256_file(output_dir / "split_manifest.csv"),
        "split_summary_sha256": sha256_file(output_dir / "split_summary.json"),
        "methods": methods,
        "near_duplicate_audit": near_info,
        "must_link_closure_repair": summary.get("benchmark_split_repair", {}),
        "rows": {"train": len(train), "val_known": len(known), "val_unknown": len(unknown)},
        "identities": {
            "train": len(train_ids),
            "val_known": len(known_ids),
            "val_unknown": len(unknown_ids),
        },
    }

def calibration_frame(split_dir: Path, data_root: Path) -> pd.DataFrame:
    """Build one canonical calibration manifest shared by both calibrations."""
    split_dir = Path(split_dir)
    parts = []
    for role, filename in (
        ("train_gallery", "train.csv"),
        ("val_known", "val_known.csv"),
        ("val_unknown", "val_unknown.csv"),
    ):
        frame = pd.read_csv(split_dir / filename, dtype=str).fillna("")
        frame["calibration_role"] = role
        parts.append(frame)
    result = pd.concat(parts, ignore_index=True)
    result["source_identity"] = result["identity"].astype(str)
    result["identity"] = [
        stable_json([str(dataset), str(identity)])
        for dataset, identity in zip(result["dataset"], result["source_identity"])
    ]
    paths = []
    for value in result["path"].astype(str):
        path = Path(value)
        if not path.is_absolute():
            path = Path(data_root) / path
        path = path.resolve()
        if not path.is_file():
            raise FileNotFoundError(f"ApexID calibration image missing: {path}")
        paths.append(str(path))
    result["path"] = paths
    result["dataset"] = "apexid_calibration"
    result["species"] = result.get("species", "unknown")
    result["image_id"] = [
        f"apexid_calibration::{hashlib.sha1(path.encode('utf-8')).hexdigest()[:20]}"
        for path in result["path"].astype(str)
    ]
    if result["image_id"].duplicated().any():
        duplicate_roles = result.loc[result["image_id"].duplicated(False), ["image_id", "calibration_role"]]
        raise RuntimeError(
            "Split V2 image occurs in multiple calibration roles: "
            f"{duplicate_roles.head().to_dict(orient='records')}"
        )
    return result.sort_values("image_id", kind="mergesort").reset_index(drop=True)

## 3. Central preparation configuration

In [ ]:
PREPARATION_VERSION = "apexid-cvpr-preparation-v2"

# Portable metadata/split output. No model embeddings are stored here.
PREP_ROOT = Path(os.environ.get(
    "APEXID_CVPR_PREP_ROOT",
    WORKSPACE_ROOT / "apexid_cvpr_prepared",
)).resolve()

MANIFEST_DIR = PREP_ROOT / "manifests"
SPLIT_DIR = PREP_ROOT / "splits"
TABLE_DIR = PREP_ROOT / "tables"
LOG_DIR = PREP_ROOT / "logs"
APEXID_SPLIT_V2_DIR = PREP_ROOT / "apexid_split_v2"

for directory in (PREP_ROOT, MANIFEST_DIR, SPLIT_DIR, TABLE_DIR, LOG_DIR, APEXID_SPLIT_V2_DIR.parent):
    directory.mkdir(parents=True, exist_ok=True)

# Large Split V2 visual-feature cache is deliberately outside the portable metadata bundle.
APEXID_SPLIT_FEATURE_CACHE = Path(os.environ.get(
    "APEXID_SPLIT_FEATURE_CACHE",
    WORKSPACE_ROOT / "apexid_split_v2_feature_cache",
)).resolve()

APEXID_DATASET_ROOT = Path(os.environ.get(
    "APEXID_DATASET_ROOT",
    "ApexID_Dataset",
)).resolve()
APEXID_METADATA_CSV = Path(os.environ.get(
    "APEXID_METADATA_CSV",
    APEXID_DATASET_ROOT / "metadata.csv",
)).resolve()

EXTERNAL_MERGED_CSV = Path(os.environ.get(
    "APEXID_EXTERNAL_MERGED_CSV",
    "external/merged.csv",
)).resolve()

BRISTOL_ROOT = Path(os.environ.get(
    "APEXID_BRISTOL_ROOT",
    "external/bristol_gorillas",
)).resolve()
BRISTOL_MANIFEST = Path(os.environ.get(
    "APEXID_BRISTOL_MANIFEST",
    BRISTOL_ROOT / "manifest.csv",
)).resolve()

JAGUAR_ROOT = Path(os.environ.get(
    "APEXID_JAGUAR_ROOT",
    "external/jaguar_antonio",
)).resolve()
JAGUAR_MANIFEST = Path(os.environ.get(
    "APEXID_JAGUAR_MANIFEST",
    JAGUAR_ROOT / "manifest.csv",
)).resolve()

JAGUAR_HF_REVISION = os.environ.get("APEXID_JAGUAR_HF_REVISION")
JAGUAR_MATERIALIZED_ROOT = Path(os.environ.get(
    "APEXID_JAGUAR_MATERIALIZED_ROOT",
    WORKSPACE_ROOT / "apexid_jaguar_materialized",
)).resolve()

EXTERNAL_DATASETS = (
    "bristol_gorillas",
    "elephants",
    "gcn_id",
    "jaguar_antonio",
)

SPLIT_SEEDS = tuple(int(value) for value in os.environ.get(
    "APEXID_SPLIT_SEEDS", "42,123,3407,2026,1337"
).split(","))

GALLERY_OBSERVATION_GROUP_RATIO = float(os.environ.get("APEXID_GALLERY_GROUP_RATIO", "0.80"))

NEAR_DUPLICATE_DHASH_THRESHOLD = int(os.environ.get("APEXID_DHASH_THRESHOLD", "4"))
NEAR_DUPLICATE_PHASH_THRESHOLD = int(os.environ.get("APEXID_PHASH_THRESHOLD", "6"))
ENCOUNTER_GAP_MINUTES = float(os.environ.get("APEXID_ENCOUNTER_GAP_MINUTES", "30"))
MINIMUM_METADATA_COVERAGE = float(os.environ.get("APEXID_MIN_METADATA_COVERAGE", "0.95"))

FINGERPRINT_WORKERS = int(os.environ.get(
    "APEXID_FINGERPRINT_WORKERS",
    str(min(8, os.cpu_count() or 1)),
))

RUN_APEXID_NEAR_DUPLICATE_AUDIT = os.environ.get(
    "APEXID_RUN_SPLIT_V2_NEAR_DUP_AUDIT", "1"
) == "1"

# Optimization switches:
# - Cache hits skip image SHA/dHash/pHash completely.
# - Set FORCE_REPREPARE=1 only after changing underlying images or preparation logic.
FORCE_REPREPARE = os.environ.get("APEXID_FORCE_REPREPARE", "0") == "1"
VERIFY_CACHED_PATHS_EXIST = os.environ.get("APEXID_VERIFY_CACHED_PATHS", "1") == "1"

seed_everything(42)
runtime_recorder = RuntimeRecorder()

PREPARATION_PROTOCOL = {
    "preparation_version": PREPARATION_VERSION,
    "split_seeds": list(SPLIT_SEEDS),
    "gallery_observation_group_ratio": GALLERY_OBSERVATION_GROUP_RATIO,
    "minimum_metadata_coverage": MINIMUM_METADATA_COVERAGE,
    "encounter_gap_minutes": ENCOUNTER_GAP_MINUTES,
    "dhash_threshold": NEAR_DUPLICATE_DHASH_THRESHOLD,
    "phash_threshold": NEAR_DUPLICATE_PHASH_THRESHOLD,
    "apexid_near_duplicate_audit": RUN_APEXID_NEAR_DUPLICATE_AUDIT,
}

USE_PREBUILT_APEXID_SPLITS = True

PREBUILT_APEXID_SPLIT_DIR = Path(
    os.environ.get("APEXID_SPLIT_ROOT", REPO_ROOT / "open_set_splits_v2")
).resolve()

print(json.dumps({
    "preparation_root": str(PREP_ROOT),
    "external_datasets": EXTERNAL_DATASETS,
    **PREPARATION_PROTOCOL,
}, indent=2))

## 4. Fail-fast input and repository preflight

In [ ]:
required = {
    "ApexID metadata": APEXID_METADATA_CSV,
    "external merged.csv": EXTERNAL_MERGED_CSV,
    "Bristol manifest": BRISTOL_MANIFEST,
}
missing = {name: str(path) for name, path in required.items() if not Path(path).is_file()}
if missing:
    raise FileNotFoundError(
        "Required benchmark inputs are missing. Edit only the configuration cell or attach the "
        f"required datasets. Missing: {json.dumps(missing, indent=2)}"
    )

if not JAGUAR_MANIFEST.is_file():
    raise FileNotFoundError(f"Mount the curated private jaguar image manifest at {JAGUAR_MANIFEST}")

from apexid.utils.provenance import capture_git_state

repo_provenance = capture_git_state(REPO_ROOT)
environment = environment_provenance()

print(json.dumps(repo_provenance, indent=2))
print("Preflight passed.")

## 5. Prepare/verify ApexID Split V2

This is the **validation/calibration side**, not the external test benchmark.  
The official repository split builder is reused, including the conservative one-way must-link
closure already present in the CVPR notebook. Existing valid Split V2 artifacts are a cache hit.

In [ ]:
# ============================================================
# Reuse finalized ApexID Split V2 directly
# ============================================================

PREBUILT_APEXID_SPLIT_DIR = Path(
    os.environ.get("APEXID_SPLIT_ROOT", REPO_ROOT / "open_set_splits_v2")
).resolve()

required_split_files = {
    "train": PREBUILT_APEXID_SPLIT_DIR / "train.csv",
    "val_known": PREBUILT_APEXID_SPLIT_DIR / "val_known.csv",
    "val_unknown": PREBUILT_APEXID_SPLIT_DIR / "val_unknown.csv",
}

missing = [
    str(path)
    for path in required_split_files.values()
    if not path.is_file()
]

if missing:
    raise FileNotFoundError(
        "Missing required prebuilt Split V2 files:\n"
        + "\n".join(missing)
    )


# ------------------------------------------------------------
# Load finalized split CSVs
# ------------------------------------------------------------

train_df = pd.read_csv(
    required_split_files["train"],
    dtype=str,
).fillna("")

val_known_df = pd.read_csv(
    required_split_files["val_known"],
    dtype=str,
).fillna("")

val_unknown_df = pd.read_csv(
    required_split_files["val_unknown"],
    dtype=str,
).fillna("")


# ------------------------------------------------------------
# Validate required columns
# ------------------------------------------------------------

required_columns = {
    "identity",
    "path",
    "dataset",
    "species",
}

for split_name, frame in {
    "train": train_df,
    "val_known": val_known_df,
    "val_unknown": val_unknown_df,
}.items():

    missing_columns = required_columns - set(frame.columns)

    if missing_columns:
        raise ValueError(
            f"{split_name}.csv is missing columns: "
            f"{sorted(missing_columns)}"
        )


# ------------------------------------------------------------
# Build globally safe identity keys
#
# Do NOT compare identity strings alone because different
# datasets may reuse the same ID string.
# ------------------------------------------------------------

def identity_keys(frame):
    return set(
        zip(
            frame["dataset"].astype(str),
            frame["identity"].astype(str),
        )
    )


train_ids = identity_keys(train_df)
val_known_ids = identity_keys(val_known_df)
val_unknown_ids = identity_keys(val_unknown_df)


# ------------------------------------------------------------
# Critical open-set invariants
# ------------------------------------------------------------

# Every known-validation identity must exist in training.
missing_known = val_known_ids - train_ids

if missing_known:
    raise RuntimeError(
        f"{len(missing_known)} val_known identities are absent "
        "from train. Example:\n"
        f"{list(sorted(missing_known))[:10]}"
    )


# Unknown validation identities must NEVER exist in training.
unknown_train_overlap = val_unknown_ids & train_ids

if unknown_train_overlap:
    raise RuntimeError(
        f"{len(unknown_train_overlap)} val_unknown identities "
        "occur in train. Example:\n"
        f"{list(sorted(unknown_train_overlap))[:10]}"
    )


# val_known and val_unknown must be identity-disjoint.
known_unknown_overlap = val_known_ids & val_unknown_ids

if known_unknown_overlap:
    raise RuntimeError(
        f"{len(known_unknown_overlap)} identities occur in both "
        "val_known and val_unknown."
    )


# ------------------------------------------------------------
# Image/path overlap check
# ------------------------------------------------------------

train_paths = set(train_df["path"].astype(str))
known_paths = set(val_known_df["path"].astype(str))
unknown_paths = set(val_unknown_df["path"].astype(str))

if train_paths & known_paths:
    raise RuntimeError(
        "The same image path occurs in both train and val_known."
    )

if train_paths & unknown_paths:
    raise RuntimeError(
        "The same image path occurs in both train and val_unknown."
    )

if known_paths & unknown_paths:
    raise RuntimeError(
        "The same image path occurs in both val_known and val_unknown."
    )


# ------------------------------------------------------------
# Optional verification of val.csv
# ------------------------------------------------------------

val_csv_path = PREBUILT_APEXID_SPLIT_DIR / "val.csv"

val_consistency_passed = None

if val_csv_path.is_file():

    val_df = pd.read_csv(
        val_csv_path,
        dtype=str,
    ).fillna("")

    expected_val = pd.concat(
        [val_known_df, val_unknown_df],
        ignore_index=True,
    )

    # Split V2 convention: val = val_known followed by val_unknown
    val_consistency_passed = val_df.reset_index(drop=True).equals(
        expected_val.reset_index(drop=True)
    )

    if not val_consistency_passed:
        raise RuntimeError(
            "val.csv is not exactly val_known.csv + val_unknown.csv."
        )


# ------------------------------------------------------------
# Copy the frozen splits into the prepared benchmark artifact
#
# Preserve an immutable copy in
# the prepared benchmark output.
# ------------------------------------------------------------

APEXID_SPLIT_V2_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

for filename in (
    "train.csv",
    "val_known.csv",
    "val_unknown.csv",
    "val.csv",
):
    source = PREBUILT_APEXID_SPLIT_DIR / filename

    if source.is_file():

        destination = APEXID_SPLIT_V2_DIR / filename

        if (
            not destination.is_file()
            or sha256_file(destination) != sha256_file(source)
        ):
            shutil.copy2(source, destination)


# ------------------------------------------------------------
# Build a reproducible split fingerprint
# ------------------------------------------------------------

split_hashes = {
    filename: sha256_file(
        APEXID_SPLIT_V2_DIR / filename
    )
    for filename in (
        "train.csv",
        "val_known.csv",
        "val_unknown.csv",
    )
}

if (APEXID_SPLIT_V2_DIR / "val.csv").is_file():
    split_hashes["val.csv"] = sha256_file(
        APEXID_SPLIT_V2_DIR / "val.csv"
    )

combined_split_fingerprint = sha256_bytes(
    json.dumps(
        split_hashes,
        sort_keys=True,
    ).encode("utf-8")
)


# ------------------------------------------------------------
# Produce the same audit object expected later in the notebook
# ------------------------------------------------------------

split_v2_audit = {
    "passed": True,
    "source": "prebuilt_finalized_split_v2",
    "source_directory": str(PREBUILT_APEXID_SPLIT_DIR),

    "train_rows": int(len(train_df)),
    "train_identities": int(len(train_ids)),

    "val_known_rows": int(len(val_known_df)),
    "val_known_identities": int(len(val_known_ids)),

    "val_unknown_rows": int(len(val_unknown_df)),
    "val_unknown_identities": int(len(val_unknown_ids)),

    "known_identity_subset_passed": True,
    "unknown_identity_disjoint_passed": True,
    "known_unknown_disjoint_passed": True,
    "path_disjointness_passed": True,

    "val_consistency_passed": val_consistency_passed,

    "split_file_sha256": split_hashes,

    # Keep this key because later notebook code expects it.
    "split_manifest_sha256": combined_split_fingerprint,
}


# ============================================================
# Build calibration manifest
# ============================================================

apexid_calibration_frame = calibration_frame(
    APEXID_SPLIT_V2_DIR,
    APEXID_DATASET_ROOT,
)

calibration_manifest_path = (
    MANIFEST_DIR / "apexid_calibration_manifest.csv"
)

atomic_write_csv(
    calibration_manifest_path,
    apexid_calibration_frame,
)


calibration_manifest_sha256 = sha256_bytes(
    (
        split_v2_audit["split_manifest_sha256"]
        +
        sha256_frame(
            apexid_calibration_frame,
            [
                "image_id",
                "path",
                "identity",
                "calibration_role",
            ],
        )
    ).encode("utf-8")
)


print("PREBUILT APEXID SPLIT V2: PASSED")
print()

print(json.dumps(
    split_v2_audit,
    indent=2,
))

print()
print(
    "Calibration manifest:",
    calibration_manifest_path,
)

print(
    "Calibration manifest fingerprint:",
    calibration_manifest_sha256,
)

## 6. Optimized preparation cache

The expensive operation here is reading every external image to compute exact/perceptual hashes.
This notebook stores a provenance record next to each canonical manifest. On a valid cache hit it
**does not reopen or rehash the images**.

When using immutable mounted datasets, trusting a matching source-manifest
signature is appropriate. If you replace image bytes while keeping the same manifest file, set
`APEXID_FORCE_REPREPARE=1`.

In [ ]:
def _read_json(path: Path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def _source_descriptor(dataset_name: str) -> dict:
    if dataset_name == "bristol_gorillas":
        return {
            "dataset": dataset_name,
            "source_type": "bristol_sam3_manifest",
            "manifest": str(BRISTOL_MANIFEST),
            "manifest_sha256": sha256_file(BRISTOL_MANIFEST),
            "root": str(BRISTOL_ROOT),
        }
    if dataset_name == "jaguar_antonio":
        return {
            "dataset": dataset_name,
            "source_type": "jaguar_prepared_manifest",
            "manifest": str(JAGUAR_MANIFEST),
            "manifest_sha256": sha256_file(JAGUAR_MANIFEST),
            "root": str(JAGUAR_ROOT),
            # Jaguar cleaning depends on ApexID identities because overlaps are removed.
            "apexid_metadata_sha256": sha256_file(APEXID_METADATA_CSV),
            "hf_revision": JAGUAR_HF_REVISION,
        }
    return {
        "dataset": dataset_name,
        "source_type": "merged_csv_filter",
        "manifest": str(EXTERNAL_MERGED_CSV),
        "manifest_sha256": sha256_file(EXTERNAL_MERGED_CSV),
        "dataset_filter": dataset_name,
        "root": str(EXTERNAL_MERGED_CSV.parent),
    }

def _external_preparation_signature(dataset_name: str) -> tuple[str, dict]:
    payload = {
        "source": _source_descriptor(dataset_name),
        "grouping_protocol": {
            "minimum_metadata_coverage": MINIMUM_METADATA_COVERAGE,
            "encounter_gap_minutes": ENCOUNTER_GAP_MINUTES,
            "dhash_threshold": NEAR_DUPLICATE_DHASH_THRESHOLD,
            "phash_threshold": NEAR_DUPLICATE_PHASH_THRESHOLD,
        },
        "preparation_version": PREPARATION_VERSION,
    }
    return sha256_bytes(stable_json(payload).encode("utf-8")), payload

def _load_cached_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, dtype=str).fillna("")

def _validate_cached_paths(frame: pd.DataFrame, dataset_name: str):
    if not VERIFY_CACHED_PATHS_EXIST:
        return
    missing = []
    for value in frame["path"].astype(str):
        if not Path(value).is_file():
            missing.append(value)
            if len(missing) >= 5:
                break
    if missing:
        raise FileNotFoundError(
            f"{dataset_name}: cached canonical manifest points to missing images: {missing}. "
            "Attach the same source dataset or force a rebuild after correcting paths."
        )

def prepare_external_dataset(dataset_name: str):
    canonical_path = MANIFEST_DIR / f"{dataset_name}_canonical_manifest.csv"
    pair_path = MANIFEST_DIR / f"{dataset_name}_near_duplicate_pairs.csv"
    record_path = MANIFEST_DIR / f"{dataset_name}_preparation_record.json"

    signature, source_payload = _external_preparation_signature(dataset_name)

    if not FORCE_REPREPARE and canonical_path.is_file() and pair_path.is_file() and record_path.is_file():
        record = _read_json(record_path)
        if (
            record.get("signature") == signature
            and record.get("canonical_manifest_sha256") == sha256_file(canonical_path)
            and record.get("near_duplicate_pairs_sha256") == sha256_file(pair_path)
        ):
            frame = _load_cached_csv(canonical_path)
            pairs = _load_cached_csv(pair_path)
            _validate_cached_paths(frame, dataset_name)
            print(f"CACHE HIT — fingerprints/groups: {dataset_name}")
            return frame, record["audit"], pairs, record["dataset_manifest_sha256"], source_payload

    # Cache miss: read source images exactly once and compute SHA/dHash/pHash in parallel.
    if dataset_name == "bristol_gorillas":
        raw_frame, audit = load_bristol_sam3(BRISTOL_MANIFEST, BRISTOL_ROOT)
    elif dataset_name == "jaguar_antonio":
        raw_frame, audit = load_jaguar_antonio(
            JAGUAR_MANIFEST,
            dataset_root=JAGUAR_ROOT,
            apexid_metadata_csv=APEXID_METADATA_CSV,
            require_black_background=True,
        )
    else:
        raw_frame, audit = load_merged_dataset(EXTERNAL_MERGED_CSV, dataset_name)

    with runtime_recorder.measure(
        "image_fingerprinting",
        dataset=dataset_name,
        number_images=len(raw_frame),
        cache_hit=False,
    ):
        fingerprinted = fingerprint_images(raw_frame, workers=FINGERPRINT_WORKERS)

    with runtime_recorder.measure(
        "observation_grouping",
        dataset=dataset_name,
        number_images=len(raw_frame),
        cache_hit=False,
    ):
        grouped, grouping_audit, pairs = build_observation_groups(
            fingerprinted,
            minimum_metadata_coverage=MINIMUM_METADATA_COVERAGE,
            encounter_gap_minutes=ENCOUNTER_GAP_MINUTES,
            dhash_threshold=NEAR_DUPLICATE_DHASH_THRESHOLD,
            phash_threshold=NEAR_DUPLICATE_PHASH_THRESHOLD,
        )

    manifest_hash = dataset_manifest_hash(grouped)
    groups_per_identity = grouped.groupby("identity")["observation_group_id"].nunique()
    protocol_ineligible_ids = set(groups_per_identity[groups_per_identity < 2].index.astype(str))

    full_audit = {**audit, **grouping_audit}
    full_audit.update({
        "final_images": int(len(grouped)),
        "final_identities": int(grouped["identity"].nunique()),
        "species": ";".join(sorted(set(grouped["species"].astype(str)))),
        "dataset_manifest_sha256": manifest_hash,
        "protocol_excluded_identities": len(protocol_ineligible_ids),
        "protocol_excluded_images": int(
            grouped["identity"].astype(str).isin(protocol_ineligible_ids).sum()
        ),
        "protocol_exclusion_reason": (
            "fewer than two observation groups; retained in canonical manifest"
        ),
    })

    atomic_write_csv(canonical_path, grouped)
    atomic_write_csv(pair_path, pairs)

    record = {
        "signature": signature,
        "source": source_payload,
        "audit": full_audit,
        "dataset_manifest_sha256": manifest_hash,
        "canonical_manifest_sha256": sha256_file(canonical_path),
        "near_duplicate_pairs_sha256": sha256_file(pair_path),
    }
    write_json(record_path, record)
    print(f"CACHE MISS — prepared fingerprints/groups: {dataset_name}")
    return grouped, full_audit, pairs, manifest_hash, source_payload

def _split_signature(dataset_manifest_sha256: str, dataset_name: str, split_seed: int) -> str:
    payload = {
        "preparation_version": PREPARATION_VERSION,
        "dataset": dataset_name,
        "dataset_manifest_sha256": dataset_manifest_sha256,
        "split_seed": int(split_seed),
        "gallery_observation_group_ratio": GALLERY_OBSERVATION_GROUP_RATIO,
    }
    return sha256_bytes(stable_json(payload).encode("utf-8"))

def prepare_split(dataset_name, frame, pairs, dataset_hash, split_seed):
    split_dir = SPLIT_DIR / dataset_name / f"seed_{split_seed}"
    split_path = split_dir / "split_manifest.csv"
    diagnostic_path = split_dir / "split_diagnostics.json"
    record_path = split_dir / "split_record.json"
    signature = _split_signature(dataset_hash, dataset_name, split_seed)

    if not FORCE_REPREPARE and split_path.is_file() and diagnostic_path.is_file() and record_path.is_file():
        record = _read_json(record_path)
        if (
            record.get("signature") == signature
            and record.get("split_manifest_sha256") == sha256_file(split_path)
        ):
            manifest = _load_cached_csv(split_path)
            check = validate_external_split(manifest, pairs)
            print(f"CACHE HIT — split: {dataset_name} / seed {split_seed}")
            return manifest, record["diagnostics"], check, record["split_manifest_sha256"]

    with runtime_recorder.measure(
        "split_generation",
        dataset=dataset_name,
        split_seed=int(split_seed),
        cache_hit=False,
    ):
        manifest, diagnostics = build_external_split_manifest(
            frame,
            split_seed=int(split_seed),
            gallery_group_ratio=GALLERY_OBSERVATION_GROUP_RATIO,
        )
        check = validate_external_split(manifest, pairs)

    split_dir.mkdir(parents=True, exist_ok=True)
    atomic_write_csv(split_path, manifest)
    write_json(diagnostic_path, {**diagnostics, "integrity": check})
    split_hash = sha256_file(split_path)
    write_json(record_path, {
        "signature": signature,
        "split_manifest_sha256": split_hash,
        "diagnostics": diagnostics,
        "integrity": check,
    })
    print(f"CACHE MISS — generated split: {dataset_name} / seed {split_seed}")
    return manifest, diagnostics, check, split_hash

## 7. Prepare external canonical manifests and observation groups

In [ ]:
# Fix the already-created RuntimeRecorder instance in the current kernel
runtime_recorder.rows = []

print(type(runtime_recorder.rows))
print(runtime_recorder.rows)

In [ ]:
external_frames = {}
near_duplicate_pairs = {}
dataset_manifest_hashes = {"apexid_calibration": calibration_manifest_sha256}
dataset_source_descriptors = {}
dataset_summary_rows = []

for dataset_name in EXTERNAL_DATASETS:
    frame, audit, pairs, manifest_hash, source_descriptor = prepare_external_dataset(dataset_name)
    external_frames[dataset_name] = frame
    near_duplicate_pairs[dataset_name] = pairs
    dataset_manifest_hashes[dataset_name] = manifest_hash
    dataset_source_descriptors[dataset_name] = source_descriptor
    dataset_summary_rows.append(audit)

dataset_summary = pd.DataFrame(dataset_summary_rows)
atomic_write_csv(TABLE_DIR / "dataset_summary_components.csv", dataset_summary)
display(dataset_summary)

## 9. Generate/freeze the five observation-aware test splits

In [ ]:
split_manifests = {}
split_manifest_hashes = {}
split_diagnostic_rows = []
split_checks = []

for dataset_name, frame in external_frames.items():
    pairs = near_duplicate_pairs.get(dataset_name)
    dataset_hash = dataset_manifest_hashes[dataset_name]

    for split_seed in SPLIT_SEEDS:
        manifest, diagnostics, check, split_hash = prepare_split(
            dataset_name,
            frame,
            pairs,
            dataset_hash,
            int(split_seed),
        )
        split_manifests[(dataset_name, int(split_seed))] = manifest
        split_manifest_hashes[(dataset_name, int(split_seed))] = split_hash
        split_diagnostic_rows.append(diagnostics)
        split_checks.append({
            "dataset": dataset_name,
            "split_seed": int(split_seed),
            **check,
        })

split_diagnostics_table = pd.DataFrame(split_diagnostic_rows)
split_checks_table = pd.DataFrame(split_checks)
atomic_write_csv(TABLE_DIR / "split_diagnostics.csv", split_diagnostics_table)
atomic_write_csv(TABLE_DIR / "split_integrity_checks.csv", split_checks_table)

display(split_diagnostics_table)

## 10. Final split integrity audit

In [ ]:
if not split_checks:
    raise RuntimeError("No external split checks were produced.")

failed = [check for check in split_checks if not bool(check.get("passed", False))]
if failed:
    raise RuntimeError(f"External benchmark integrity failed: {failed[:3]}")

# Re-read and revalidate saved split files, not merely in-memory objects.
for (dataset_name, split_seed), expected_hash in split_manifest_hashes.items():
    split_path = SPLIT_DIR / dataset_name / f"seed_{split_seed}" / "split_manifest.csv"
    if sha256_file(split_path) != expected_hash:
        raise RuntimeError(f"Saved split hash changed unexpectedly: {split_path}")
    saved = _load_cached_csv(split_path)
    validate_external_split(saved, near_duplicate_pairs.get(dataset_name))

if not split_v2_audit.get("passed", False):
    raise RuntimeError("ApexID Split V2 did not pass validation.")

print(f"Verified {len(split_checks)} external split manifests.")
print("ApexID Split V2: PASSED")
print("External observation-aware split integrity: PASSED")

## 11. Freeze, hash, and export

This cell creates a lightweight portable ZIP of **metadata/splits/provenance only**.  
It intentionally does not copy all source images or the potentially large Split V2 feature cache.

Preserve the completed bundle with its checksums.
Future model notebooks should attach this prepared bundle **plus the same source image datasets**.

In [ ]:
# Save timing first.
runtime_recorder.save(LOG_DIR / "preparation_runtime.csv")

preparation_manifest = {
    "preparation_version": PREPARATION_VERSION,
    "source_benchmark_version": BENCHMARK_VERSION,
    "status": "passed",
    "repository": repo_provenance,
    "environment": environment,
    "protocol": PREPARATION_PROTOCOL,
    "inputs": {
        "apexid_dataset_root": str(APEXID_DATASET_ROOT),
        "apexid_metadata_csv": str(APEXID_METADATA_CSV),
        "apexid_metadata_sha256": sha256_file(APEXID_METADATA_CSV),
        "external_sources": dataset_source_descriptors,
        "jaguar_materialized_locally": str(JAGUAR_MATERIALIZED_ROOT)
            if JAGUAR_MATERIALIZED_ROOT.exists()
            and str(JAGUAR_ROOT).startswith(str(JAGUAR_MATERIALIZED_ROOT))
            else None,
    },
    "apexid_split_v2": split_v2_audit,
    "apexid_calibration_manifest": {
        "path": str(calibration_manifest_path),
        "sha256": sha256_file(calibration_manifest_path),
        "logical_fingerprint": calibration_manifest_sha256,
    },
    "dataset_manifest_hashes": dataset_manifest_hashes,
    "dataset_manifest_file_hashes": {
        dataset: sha256_file(MANIFEST_DIR / f"{dataset}_canonical_manifest.csv")
        for dataset in EXTERNAL_DATASETS
    },
    "split_manifest_hashes": {
        f"{dataset}::seed_{seed}": digest
        for (dataset, seed), digest in sorted(split_manifest_hashes.items())
    },
    "integrity": {
        "apexid_split_v2_passed": True,
        "external_split_checks": len(split_checks),
        "external_split_checks_passed": all(
            bool(check.get("passed")) for check in split_checks
        ),
    },
}
write_json(PREP_ROOT / "preparation_manifest.json", preparation_manifest)

reuse_readme = f"""# ApexID CVPR Prepared Benchmark

Preparation version: `{PREPARATION_VERSION}`

This directory contains frozen metadata/manifests/splits only. No model checkpoint or embedding is required.

## Future model workflow

1. Attach the same source image datasets.
2. Load `preparation_manifest.json` and verify the artifact hashes.
3. Extract closed-set embeddings from the canonical external catalogues.
4. Reuse the gallery/query manifests under splits/.
5. For AnimalCLEF, use the independent open-set roles and WildFusion calibration cohort.

Gallery observation-group ratio: {GALLERY_OBSERVATION_GROUP_RATIO}
Split seeds: {list(SPLIT_SEEDS)}
dHash threshold: {NEAR_DUPLICATE_DHASH_THRESHOLD}
pHash threshold: {NEAR_DUPLICATE_PHASH_THRESHOLD}

The benchmark is valid only if `preparation_manifest.json` reports `status = passed`.
"""
atomic_write_text(PREP_ROOT / "README_NEXT_MODEL.md", reuse_readme)

# Index the completed metadata/split tree. Source images are not traversed or rehashed.
artifact_rows = []
for artifact_path in sorted(PREP_ROOT.rglob("*")):
    if artifact_path.is_file() and artifact_path.name != "artifact_index.csv":
        artifact_rows.append({
            "relative_path": str(artifact_path.relative_to(PREP_ROOT)),
            "bytes": int(artifact_path.stat().st_size),
            "sha256": sha256_file(artifact_path),
        })
artifact_index = pd.DataFrame(artifact_rows)
atomic_write_csv(PREP_ROOT / "artifact_index.csv", artifact_index)

# Lightweight portable bundle containing metadata/splits/provenance only.
# Source images and the potentially large Split V2 feature cache are not copied.
bundle_base = WORKSPACE_ROOT / "apexid_cvpr_prepared_bundle"
bundle_zip = Path(shutil.make_archive(
    str(bundle_base),
    "zip",
    root_dir=PREP_ROOT.parent,
    base_dir=PREP_ROOT.name,
))

print("PREPARATION INTEGRITY CHECK: PASSED")
print("Prepared root:", PREP_ROOT)
print("Portable bundle:", bundle_zip)
print("Calibration manifest:", calibration_manifest_path)
print("External split manifests:", len(split_manifest_hashes))

The bundle stores split manifests, checksums, and image metadata. Mount the same foreground images for extraction. When paths change, use a prefix mapping as described in the evaluation guide.